# Patent Topic Modeling with TF‑IDF + NMF


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
import re
from pathlib import Path


## Step 1 — Load and Combine Text

In [ ]:
# Note this csv is saved to the Drive as well
df = pd.read_csv("../data/patents/v1_core_expansion/core/claims_added/patents_with_claims.csv")

print(f"Dataset loaded: {df.shape[0]} patents")

# Convert to list for processing
docs = df["claims"].astype(str).tolist()


Dataset loaded: 38284 patents


## Step 2 — Remove Patent + GPU Boilerplate

In [ ]:
custom_stopwords = [

    # Claim/legal glue
    "claim","claims","claimed","recited","reciting",
    "wherein","whereby",
    "comprising","comprises","comprised",
    "including","includes","include",
    "consisting","consist","consists",

    #Candidate ranks
    "social","networking","online","profile","profiles","candidate","candidates",
    "job","ranking","ranked","query","queries","search","member","members","service","services",
    "score","scores", "assistant", "automated", "content",
    "user", "client", "interaction",
    "language", "text", "audio",
    "search", "query", "profile",
    "candidate", "job",
    "media", "item",

    #Telecom
    "wireless", "radio",
    "ue", "equipment",
    "user equipment",
    "base station",
    "transmit", "transmitting",
    "transmission",
    "channel",
    "station" 

    # Structural quantifiers
    "plurality","first","second","third","fourth","fifth",
    "one","two","at","least",

    # Reference words
    "said","such","thereof","therein","thereon",
    "therewith","thereto","thereby","thereafter",
    "corresponding","associated","respective",

    # Boilerplate nouns
    "embodiment","embodiments",
    "aspect","aspects",
    "disclosure","present",
    "example","examples",

    # Functional filler verbs
    "configured","configure","configuring",
    "adapted","adapt",
    "operable","operate","operating",
    "may","can","could",
    "based",
    "perform","performs","performing",
    "receive","receives","receiving",
    "determine","determines","determining",

    # Diagram/figure noise
    "figure","figures","fig","figs",
    "shown","illustrated","illustrates","illustrating",
    "depicted","depicts","depicting",
    "diagram","diagrams","drawing","drawings",
    "chart","charts","schematic","schematics",
    "flowchart","flowcharts",

    # GPU identity (remove to expose substructure)
    "gpu","gpus","graphics","graphic",

    # Very generic nouns
    "method","apparatus","system","device","devices",
]

stop_words = set(ENGLISH_STOP_WORDS).union(set(custom_stopwords))


def clean_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^a-z0-9_\s\-]", " ", s)         # keep alnum/_/- as token-friendly
    s = re.sub(r"\b\d{1,2}\b", " ", s)             # remove standalone 1-2 digit numbers
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_clean"] = df["claims"].map(clean_text)

# Quick sanity check
df[["claims","text_clean"]].head(3)

,claims,text_clean
0,['1 . A processor comprising: a plurality of c...,a processor comprising a plurality of cores an...
1,['1 . A graphics processor for a multi-tile ar...,a graphics processor for a multi-tile architec...
2,"['1 . A method, comprising: determining whethe...",a method comprising determining whether a port...


## Step 3 — Vectorize with TF‑IDF

In [ ]:
stop_words = set(w.lower() for w in stop_words) # This is likely unnecessary but keeping to be safe

In [ ]:

vectorizer = TfidfVectorizer(
    stop_words=list(stop_words),
    ngram_range=(1, 2),
    min_df=20, # had to raise this to avoid blowing up the vocab
    max_df=0.3, # ignore common terms
    lowercase=True,
    sublinear_tf=True,        
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_]{1,}\b"  # >=2 chars, starts with letter
)

X = vectorizer.fit_transform(df["text_clean"])
feature_names = np.array(vectorizer.get_feature_names_out())

print(f"TF‑IDF matrix shape: {X.shape}  (docs × terms)")
print(f"Vocabulary size: {len(feature_names):,}")


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['base', 'station'] not in stop_words.
  warnings.warn(


TF‑IDF matrix shape: (38284, 52948)  (docs × terms)
Vocabulary size: 52,948


## Step 4 — Fit NMF Topic Model

In [ ]:
# NMF hyperparameters
n_topics = 18 

nmf = NMF(
    n_components=n_topics,
    init="nndsvda",
    random_state=42,
    max_iter=1000,
    alpha_W=0,        # regularization on W (doc-topic)
    alpha_H=0.3        # regularization on H (topic-term) forces topics to be more distinct
)

W = nmf.fit_transform(X)  # doc-topic matrix
H = nmf.components_       # topic-term matrix

print(f"W shape: {W.shape} (docs × topics)")
print(f"H shape: {H.shape} (topics × terms)")
print(f"Reconstruction error: {nmf.reconstruction_err_:.4f}")


W shape: (38284, 18) (docs × topics)
H shape: (18, 52948) (topics × terms)
Reconstruction error: 190.6938


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


## Step 5 — Inspect Topics

In [ ]:
def print_top_words(H, feature_names, n_top_words=15):
    for topic_idx, topic in enumerate(H):
        top = np.argsort(topic)[::-1][:n_top_words]
        words = feature_names[top]
        weights = topic[top]
        print(f"Topic {topic_idx:02d}: " + ", ".join([f"{w}" for w in words]))

print_top_words(H, feature_names, n_top_words=20)


Topic 00: model, learning, machine learning, learning model, machine, training, trained, training data, model trained, models, features, ml, train, model using, training machine, trained machine, using machine, input, ml model, parameters
Topic 01: cache, cache line, line, cache memory, lines, cache lines, level cache, level, data cache, cache cache, request, memory cache, line cache, cache controller, tag, miss, entry, plurality cache, processor cache, core
Topic 02: operand, source, instruction, destination, register, packed, data element, data elements, element, packed data, elements, decode, decoded, source operand, bit, opcode, execution, execution circuitry, vector, decoded instruction
Topic 03: pixel, shader, primitive, tile, primitives, vertex, pixels, rendering, shading, texture, sample, pipeline, tiles, geometry, vertices, render, samples, buffer, depth, processing unit
Topic 04: virtual, resource, computing, virtual machine, resources, request, application, instance, machine

In [74]:
import numpy as np
import itertools
from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# CONFIG
# -----------------------------
k_values = [16, 18, 20, 22, 24, 26, 28]
seeds = [0,1,2,3,4]
topn = 20  # top words per topic for stability check

results = []

def top_word_sets(H, feature_names, topn=20):
    sets = []
    for k in range(H.shape[0]):
        top = np.argsort(H[k])[::-1][:topn]
        sets.append(set(feature_names[top]))
    return sets

def jaccard(a, b):
    return len(a & b) / max(1, len(a | b))

def average_best_jaccard(sets1, sets2):
    scores = []
    for s1 in sets1:
        best = max(jaccard(s1, s2) for s2 in sets2)
        scores.append(best)
    return np.mean(scores)

# -----------------------------
# SWEEP
# -----------------------------
for n_topics in k_values:
    print(f"\n===== k = {n_topics} =====")
    
    seed_topic_sets = []
    reconstruction_errors = []
    redundancy_scores = []
    
    for seed in seeds:
        
        nmf = NMF(
            n_components=n_topics,
            init="nndsvda",
            random_state=seed,
            max_iter=500,
            alpha_W=0,
            alpha_H=0.3
        )
        
        W = nmf.fit_transform(X)
        H = nmf.components_
        
        reconstruction_errors.append(nmf.reconstruction_err_)
        
        # ---- redundancy (topic similarity within same run)
        S = cosine_similarity(H)
        np.fill_diagonal(S, np.nan)
        redundancy_scores.append({
            "max_sim": np.nanmax(S),
            "mean_sim": np.nanmean(S)
        })
        
        # ---- store topic word sets for stability
        topic_sets = top_word_sets(H, feature_names, topn=topn)
        seed_topic_sets.append(topic_sets)
    
    # ---- stability across seeds
    pairwise_stabilities = []
    for s1, s2 in itertools.combinations(seed_topic_sets, 2):
        pairwise_stabilities.append(average_best_jaccard(s1, s2))
    
    avg_stability = np.mean(pairwise_stabilities)
    
    # ---- aggregate redundancy
    avg_max_sim = np.mean([r["max_sim"] for r in redundancy_scores])
    avg_mean_sim = np.mean([r["mean_sim"] for r in redundancy_scores])
    
    print(f"Avg Reconstruction Error: {np.mean(reconstruction_errors):.4f}")
    print(f"Avg Stability (Jaccard): {avg_stability:.3f}")
    print(f"Avg Max Topic Cosine: {avg_max_sim:.3f}")
    print(f"Avg Mean Topic Cosine: {avg_mean_sim:.3f}")
    
    results.append({
        "k": n_topics,
        "recon_error": np.mean(reconstruction_errors),
        "stability": avg_stability,
        "max_cosine": avg_max_sim,
        "mean_cosine": avg_mean_sim
    })

# -----------------------------
# SUMMARY TABLE
# -----------------------------
import pandas as pd
results_df = pd.DataFrame(results)
print("\n===== SUMMARY =====")
print(results_df.sort_values("k"))


===== k = 16 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve 

Avg Reconstruction Error: 190.9146
Avg Stability (Jaccard): 0.877
Avg Max Topic Cosine: 0.276
Avg Mean Topic Cosine: 0.116

===== k = 18 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve 

Avg Reconstruction Error: 190.6959
Avg Stability (Jaccard): 0.940
Avg Max Topic Cosine: 0.279
Avg Mean Topic Cosine: 0.107

===== k = 20 =====


...

/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve 

Avg Reconstruction Error: 189.9657
Avg Stability (Jaccard): 0.789
Avg Max Topic Cosine: 0.290
Avg Mean Topic Cosine: 0.080

===== k = 28 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


Avg Reconstruction Error: 189.7991
Avg Stability (Jaccard): 0.740
Avg Max Topic Cosine: 0.311
Avg Mean Topic Cosine: 0.076

===== SUMMARY =====
    k  recon_error  stability  max_cosine  mean_cosine
0  16   190.914566   0.877306    0.276293     0.115753
1  18   190.695920   0.940175    0.278893     0.106656
2  20   190.501016   0.876430    0.288213     0.098922
3  22   190.314615   0.833533    0.318753     0.092625
4  24   190.139633   0.761343    0.300990     0.084832
5  26   189.965744   0.788785    0.290365     0.079735
6  28   189.799122   0.739805    0.310903     0.075812


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
